In [1]:
import csv
from rdflib import Graph, Dataset, Namespace, URIRef, BNode,RDF, RDFS, XSD, Literal, ConjunctiveGraph
from rdflib.namespace import RDF, split_uri
from collections import defaultdict

In [2]:
saref = Namespace("https://saref.etsi.org/core/")
s4ener = Namespace("https://saref.etsi.org/saref4ener/")
s4bldg = Namespace("https://saref.etsi.org/saref4bldg/")
geo = Namespace("http://www.opengis.net/ont/geosparql#")
EX = Namespace("https://example.org/")

In [3]:
g = ConjunctiveGraph()
g.parse("data.2026-01-26.log", format="trig")

C:\Users\gin\AppData\Local\Temp\ipykernel_23880\1165208509.py:1: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


<Graph identifier=Nf183ebc4ba864dab981d73b4608d7838 (<class 'rdflib.graph.ConjunctiveGraph'>)>

In [4]:
print("Statements in total:", len(g))

Statements in total: 4158741


In [5]:
print("Number of graphs:", len(list(g.contexts())))

Number of graphs: 336998


In [6]:
with open("meter_locations.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        meter_uri = row["meter_uri"]
        meter_long = float(row["meter_long"])
        meter_lat = float(row["meter_lat"])

        meter = URIRef(meter_uri)
        meter_geom = BNode()

        g.add((meter, RDF.type, saref.Meter))
        g.add((meter, geo.hasGeometry, meter_geom))

        wkt = f"POINT({meter_long} {meter_lat})"
        g.add((meter_geom, geo.asWKT, Literal(wkt)))

        bldgs = row["metered_location"].split("/")

        for b in bldgs:
            bldg = URIRef(EX[b.strip()])
            g.add((meter, EX.measuresBuilding, bldg))

In [7]:
with open("kadaster_buildings.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        gebouw = row["gebouw"]
        build = row["build"]

        bldg = EX[build]
        bldg_geom = BNode()

        g.add((bldg, RDF.type, s4bldg.Building))
        g.add((bldg, geo.hasGeometry, bldg_geom))
        g.add((bldg_geom, geo.asWKT, Literal(gebouw)))

In [8]:
# Which building is meter X measuring?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX EX: <https://example.org/>

SELECT ?meter ?building WHERE {
  ?meter a saref:Meter ;
         EX:measuresBuilding ?building .
}
"""

for meter_uri, bldg_uri in g.query(q):
    meter_name = meter_uri.split("meter/")[1]
    _, bldg_name = split_uri(bldg_uri)
    
    print(meter_name, "measures building", bldg_name)

scu200/SCU-200-B16/18 measures building B18
scu200/SCU-200-B16/19 measures building B19
scu200/SCU-200-B09/31 measures building B09
scu200/SCU-200-B09/32 measures building B01
scu200/SCU-200-B09/33 measures building B11
scu200/SCU-200-B09/34 measures building B40
scu200/SCU-200-B09/35 measures building B09
scu200/SCU-200-B09/38 measures building B08
scu200/SCU-200-B09/40 measures building B02
scu200/SCU-200-B46/41 measures building B38
scu200/SCU-200-B46/42 measures building B46
scu200/SCU-200-B46/43 measures building B46
scu200/SCU-200-B46/44 measures building B46
scu200/SCU-200-B31-2/45 measures building B31
scu200/SCU-200-B31-2/46 measures building B20
scu200/SCU-200-B31-2/47 measures building B20
scu200/SCU-200-B31-2/48 measures building B22
scu200/SCU-200-B31-2/49 measures building B32
scu200/SCU-200-B31-1/50 measures building B31
scu200/SCU-200-B31-1/51 measures building B48
scu200/SCU-200-B31-1/51 measures building B54
scu200/SCU-200-B31-1/52 measures building B31
BCU105420 meas

In [9]:
# Is there a meter without a location?

q ="""
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX EX: <https://example.org/>

SELECT ?meter WHERE {
  ?meter a saref:Meter .
  FILTER NOT EXISTS {
    ?meter EX:measuresBuilding ?building .
  }
}
"""

for (meter_uri,) in g.query(q):
    meter_name = meter_uri.split("meter/")[1]
    
    print(meter_name, "measures no buildings")

In [10]:
# Is there a building without a meter?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX EX: <https://example.org/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
  FILTER NOT EXISTS {
    ?meter EX:measuresBuilding ?building
  }

}
"""

for (bldg_uri,) in g.query(q):
    _, bldg_name = split_uri(bldg_uri)
    print(bldg_name)

B50
B52
B04
B13
B17
B30
B44
B14
M01


In [11]:
# Which meters are measuring the same building?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX EX: <https://example.org/>

    SELECT ?meter WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
    }}
    """

    if len(g.query(q2)) > 1:
      _, bldg_name = split_uri(bldg_uri)
      print("In building",bldg_name,":")

      for (meter_uri,) in g.query(q2):
        meter_name = meter_uri.split("meter/")[1]

        print(meter_name)

In building B09 :
scu200/SCU-200-B09/31
scu200/SCU-200-B09/35
In building B12 :
tap/tap-kb/B12-cp01
tap/tap-kb/B12-cp02
tap/tap-kb/B12-cp03
tap/tap-kb/B12-cp04
In building B16 :
871687120000000080/6500036557
871687120000092214/6500071036
In building B20 :
scu200/SCU-200-B31-2/46
scu200/SCU-200-B31-2/47
In building B31 :
scu200/SCU-200-B31-2/45
scu200/SCU-200-B31-1/50
scu200/SCU-200-B31-1/52
netx/bms-opc-ua/IPS-S3.1.1-IP-interface-DIN-railapparaat
netx/bms-opc-ua/FBXi-8R8-331123
netx/bms-opc-ua/AC-S1.2.1-Application-Controller
In building B46 :
scu200/SCU-200-B46/42
scu200/SCU-200-B46/43
scu200/SCU-200-B46/44
In building B42 :
BCU105420
BCU105405
BCU105369
BCU109794
BCU105423
BCU105418
BCU105425
BCU105404
871687110003209825
3266975


In [ ]:
# How do daily energy measurement patterns between buildings X and Y differ?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value ?property WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn ?property .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)
        print(bldg_name,":")
        bldg_kWh = [0, 0, 0, 0]
        bldg_A1 = []
        bldg_A2 = []
        bldg_A3 = []
        bldg_V1 = []
        bldg_V2 = []
        bldg_V3 = []
        bldg_V4 = []
        bldg_V5 = []
        bldg_V6 = []
        bldg_W1 = []
        bldg_W2 = []
        bldg_W3 = []
        bldg_W4 = []

        meter_vals = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        for meter_uri, time, value, prpt_uri in result:
            minute = str(time)[:16]
            meter_vals[meter_uri][prpt_uri][minute].append(float(value))

        for meter_uri, properties in meter_vals.items():
            for prpt_uri, times in properties.items():
                for time, values in times.items():
                    prpt_name = prpt_uri.split("/")[-1]
                    
                    if prpt_name == "KiloW-HR":
                        first_a, first_b, first_c, first_d = values[:4]
                        last_a, last_b, last_c, last_d = values[-4:]

                        usage = [
                            last_a - first_a,
                            last_b - first_b,
                            last_c - first_c,
                            last_d - first_d
                        ]

                        for i in range(4):
                            bldg_kWh[i] += usage[i]

                    elif prpt_name == "A":
                        A1 = []
                        A2 = []
                        A3 = []

                        for i in range(0, len(values), 3):
                            chunk = values[i:i+3]

                            if len(chunk) != 3:
                                print("Broken A meter:")
                                print(meter_uri)
                                print("Chunk length:", len(chunk))
                                print("Remaining values:", chunk)
                                continue

                            A1.append(chunk[0])
                            A2.append(chunk[1])
                            A3.append(chunk[2])

                        bldg_A1.extend(A1)
                        bldg_A2.extend(A2)
                        bldg_A3.extend(A3)

                    elif prpt_name == "V":
                        V1 = []
                        V2 = []
                        V3 = []
                        V4 = []
                        V5 = []
                        V6 = []

                        for i in range(0, len(values), 6):
                            chunk = values[i:i+6]

                            if len(chunk) != 6:
                                print("Broken V meter:")
                                print(meter_uri)
                                print("Chunk length:", len(chunk))
                                print("Remaining values:", chunk)
                                continue

                            V1.append(chunk[0])
                            V2.append(chunk[1])
                            V3.append(chunk[2])
                            V4.append(chunk[3])
                            V5.append(chunk[4])
                            V6.append(chunk[5])

                        bldg_V1.extend(V1)
                        bldg_V2.extend(V2)
                        bldg_V3.extend(V3)
                        bldg_V4.extend(V4)
                        bldg_V5.extend(V5)
                        bldg_V6.extend(V6)

                    elif prpt_name == "W":
                        W1 = []
                        W2 = []
                        W3 = []
                        W4 = []
                        
                        for i in range(0, len(values), 4):
                            chunk = values[i:i+4]

                            if len(chunk) != 4:
                                print("Broken W meter:")
                                print(meter_uri)
                                print("Chunk length:", len(chunk))
                                print("Remaining values:", chunk)
                                continue

                            W1.append(chunk[0])
                            W2.append(chunk[1])
                            W3.append(chunk[2])
                            W4.append(chunk[3])

                        bldg_W1.extend(W1)
                        bldg_W2.extend(W2)
                        bldg_W3.extend(W3)
                        bldg_W4.extend(W4)

        bldg_A = [
            sum(bldg_A1) / len(bldg_A1),
            sum(bldg_A2) / len(bldg_A2),
            sum(bldg_A3) / len(bldg_A3)
        ]
        
        bldg_V = [
            sum(bldg_V1) / len(bldg_V1),
            sum(bldg_V2) / len(bldg_V2),
            sum(bldg_V3) / len(bldg_V3),
            sum(bldg_V4) / len(bldg_V4),
            sum(bldg_V5) / len(bldg_V5),
            sum(bldg_V6) / len(bldg_V6)
        ]
        
        bldg_W = [
            sum(bldg_W1) / len(bldg_W1),
            sum(bldg_W2) / len(bldg_W2),
            sum(bldg_W3) / len(bldg_W3),
            sum(bldg_W4) / len(bldg_W4)
        ]

        print(bldg_V, bldg_A, bldg_W, bldg_kWh)

B01 :
Broken W meter:
https://cgi.com/hedgeiot/meter/scu200/SCU-200-B09/32
Chunk length: 2
Remaining values: [3155.6, 2763.75]
[312.96880530973453, 345.9627581120944, 273.1740412979351, 320.40184365781715, 288.0307522123894, 360.50929203539823] [28.289181415929203, 33.905501474926254, 29.765117994100294] [10135.11610619469, 7388.046467551622, 13569.652411504427, 8670.732219764011] [476.1699999999837, 143.42000000000553, 175.89999999999418, 156.84000000001106]
B02 :
Broken V meter:
https://cgi.com/hedgeiot/meter/scu200/SCU-200-B09/40
Chunk length: 2
Remaining values: [401.90000000000003, 402.20000000000005]
[332.8543046357616, 344.21383370125096, 304.9150110375276, 298.6301692420898, 287.4974981604121, 327.50647534952174] [1.4720250368188512, 1.1492047128129603, 1.8013107511045654] [427.40155375552285, 337.5682032400589, 296.62629602356407, 267.6206995581738] [15.909999999999854, 7.180000000000291, 4.350000000000364, 4.380000000000109]
B09 :
Broken V meter:
https://cgi.com/hedgeiot/mete

In [13]:
# What is the aggregated energy usage of a building over a day?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/KiloW-HR> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)
        print(bldg_name,":")
        bldg_usage = [0, 0, 0, 0]

        meter_values = defaultdict(list)
        for meter_uri, time, value in result:
            meter_values[meter_uri].append(float(value))

        for meter_uri, values in meter_values.items():
            first_a, first_b, first_c, first_d = values[:4]
            last_a, last_b, last_c, last_d = values[-4:]

            usage = [
                last_a - first_a,
                last_b - first_b,
                last_c - first_c,
                last_d - first_d
            ]

            for i in range(4):
                bldg_usage[i] += usage[i]

        for stream in bldg_usage:
            print(stream,"kWh")

B01 :
476.1699999999837 kWh
143.42000000000553 kWh
175.89999999999418 kWh
156.84000000001106 kWh
B02 :
15.909999999999854 kWh
7.180000000000291 kWh
4.350000000000364 kWh
4.380000000000109 kWh
B09 :
112.73000000000229 kWh
25.570000000000164 kWh
27.909999999999997 kWh
59.25999999999931 kWh
B11 :
135.43000000000757 kWh
44.45999999999913 kWh
52.81000000000131 kWh
38.159999999999854 kWh
B18 :
91.60000000000036 kWh
31.409999999999854 kWh
22.75999999999999 kWh
37.440000000000055 kWh
B19 :
0.0 kWh
0.0 kWh
0.0 kWh
0.0 kWh
B20 :
26.029999999999745 kWh
10.720000000000027 kWh
7.720000000000027 kWh
7.579999999999927 kWh
B31 :
314.0400000000009 kWh
121.61999999999898 kWh
104.54999999999927 kWh
87.88000000000102 kWh
B40 :
36.43000000000029 kWh
1.0300000000002 kWh
7.539999999999964 kWh
27.86999999999989 kWh


In [ ]:
# Can meter location be visualised on a map with markings being proportional to aggregated usage?

import folium

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX s4ener: <https://saref.etsi.org/saref4ener/>
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX EX: <https://example.org/>

SELECT ?building WHERE {
  
}
"""

In [ ]:
# Which building shows the highest average power consumption over an hour?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX EX: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter EX:measuresBuilding <{bldg_uri}> .
      FILTER NOT EXISTS{{
        ?meter a saref4ab:ChargePointMeter
      }}
      FILTER STRSTARTS(
        STR(?meter),
        "https://cgi.com/hedgeiot/meter/"
      )
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/W> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    

In [ ]:
g.serialize(destination="enriched_ontology.log", format="trig")